# 01 - Exploración de la API NASA NeoWs

Exploración inicial de la API NeoWs (Near Earth Object Web Service) de NASA.

**Objetivo:** Entender la estructura de los datos antes de construir el pipeline ETL.

## Configuración y conexión

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import time

load_dotenv()
API_KEY = os.getenv("NASA_API_KEY")

# Verificar que la key cargó bien
print("API Key cargada:", API_KEY is not None)

url = "https://api.nasa.gov/neo/rest/v1/feed"

params = {
    "start_date" : "2026-04-01",
    "end_date" : "2026-04-01",
    "api_key" : API_KEY
}

response = requests.get(url, params = params)

print(response.status_code)
print(type(response.json()))
print(response.json().keys())

API Key cargada: True
200
<class 'dict'>
dict_keys(['links', 'element_count', 'near_earth_objects'])


## Estructura de los datos

Inspeccionamos qué devuelve la API para un día específico y qué campos 
tiene cada asteroide.

In [2]:
data = response.json()

# Cuántos asteroides encontró ese día
print("Asteroides ese día:", data['element_count'])

# La data viene organizada por fecha
objetos_por_fecha = data['near_earth_objects']
print("Fechas en el response:", list(objetos_por_fecha.keys()))

# Datos del primer asteriode
primer_asteroide = objetos_por_fecha['2026-04-01'][0]
print("\nCampos disponibles:")
for key, value in primer_asteroide.items():
    print(f"  {key}: {value}")

Asteroides ese día: 16
Fechas en el response: ['2026-04-01']

Campos disponibles:
  links: {'self': 'http://api.nasa.gov/neo/rest/v1/neo/3802103?api_key=p1HfHwH61FZc2aNYvQs7Zb5knPStE0Ds4pHb0JIn'}
  id: 3802103
  neo_reference_id: 3802103
  name: (2018 FK5)
  nasa_jpl_url: https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=3802103
  absolute_magnitude_h: 28.3
  estimated_diameter: {'kilometers': {'estimated_diameter_min': 0.0058150704, 'estimated_diameter_max': 0.0130028927}, 'meters': {'estimated_diameter_min': 5.8150703965, 'estimated_diameter_max': 13.0028927004}, 'miles': {'estimated_diameter_min': 0.0036133161, 'estimated_diameter_max': 0.0080796204}, 'feet': {'estimated_diameter_min': 19.0783155595, 'estimated_diameter_max': 42.6604104873}}
  is_potentially_hazardous_asteroid: False
  close_approach_data: [{'close_approach_date': '2026-04-01', 'close_approach_date_full': '2026-Apr-01 21:06', 'epoch_date_close_approach': 1775077560000, 'relative_velocity': {'kilometers_per_seco

## Campos seleccionados para el modelo

De todos los campos disponibles, usaremos:

| Campo | Descripción |
|-------|-------------|
| `absolute_magnitude_h` | Brillo del asteroide — indirectamente indica tamaño |
| `diameter_min_km` / `diameter_max_km` | Tamaño estimado en km |
| `velocity_km_per_hour` | Velocidad de acercamiento |
| `miss_distance_km` | Distancia mínima a la Tierra |
| `is_potentially_hazardous` | **Target** — lo que el modelo va a predecir |

El ETL completo está en `02_etl.ipynb`.